# 🎤 Clone Your Own Voice (Free) — ClarityVoice

This notebook uses the free, open-source **XTTS-v2** model to make speech in **your** voice, from a short recording of you talking.

### How to use
1. **Turn on the free GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.
2. Run each cell **top to bottom** (click the ▶ on the left of each cell).
3. When asked, upload a **clear 15–30 second recording of your own voice** (WAV or MP3, no background noise — just you talking naturally).
4. Type what you want it to say, run the last cell, and it plays + downloads your cloned voice.

> First run downloads the model (~2 min). That's normal.

### 1. Check the GPU is on

In [ ]:
!nvidia-smi -L
# If this says 'command not found' or shows no GPU, go to Runtime > Change runtime type > T4 GPU.

### 2. Install the voice library (~1 min)

In [ ]:
!pip install -q coqui-tts "transformers==4.49.0"
print("Installed. Now do Runtime > Restart session, then run from cell 3 (skip this install).")

### 3. Load the model

In [ ]:
import os, torch
os.environ["COQUI_TOS_AGREED"] = "1"

# Make XTTS load on new PyTorch. Patch ONCE (safe to re-run — avoids RecursionError):
if not getattr(torch, "_xtts_patched", False):
    _real_torch_load = torch.load
    def _safe_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _real_torch_load(*args, **kwargs)
    torch.load = _safe_load
    torch._xtts_patched = True

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded ✅")

### 4. Upload YOUR voice recordings
**Name each file with its language:** `english1.m4a`, `english2.m4a`, `hindi1.m4a`, `hindi2.m4a`.
Keeping English and Hindi clips **separate** gives a cleaner voice in each language — mixing different-accent clips into one blend muddies the result.

In [ ]:
from google.colab import files
print("Upload your clips. Name them by language: english1.m4a, english2.m4a, hindi1.m4a, hindi2.m4a")
uploaded = files.upload()
all_files = list(uploaded.keys())

# Keep languages SEPARATE so each accent stays clean (mixing them muddies the voice).
en_samples = [f for f in all_files if "eng" in f.lower() or f.lower().startswith("en")]
hi_samples = [f for f in all_files if "hin" in f.lower() or "urdu" in f.lower() or f.lower().startswith("hi")]
if not en_samples: en_samples = all_files   # fallback if naming didn't match
if not hi_samples: hi_samples = all_files
ref = {"en": en_samples, "hi": hi_samples}

print("\nEnglish reference clips:", en_samples)
print("Hindi reference clips:  ", hi_samples)

### 5. Make it speak in your voice — multiple languages ✨
Each line below is one language. Edit the text, then run. This model supports **English (`en`)** and **Hindi (`hi`)**. Kannada is **not** supported here (ask for the Indic model for Kannada).

In [ ]:
from IPython.display import Audio, display

# Natural, well-punctuated text + language-matched clips + low temperature = cleanest result.
clips = {
    "en": "Hello, this is my own voice. I think it sounds clear and natural.",
    "hi": "नमस्ते, यह मेरी अपनी आवाज़ है। मुझे लगता है यह साफ़ और स्वाभाविक लगती है।",
}

for lang, text in clips.items():
    out = f"my_voice_{lang}.wav"
    tts.tts_to_file(
        text=text, speaker_wav=ref[lang], language=lang, file_path=out,  # language-matched, clean
        temperature=0.7,          # low = faithful to YOUR voice, fewer artifacts
        repetition_penalty=2.0,
        top_k=50, top_p=0.8,
        length_penalty=1.0,
        speed=1.0,
        enable_text_splitting=True,
    )
    print(f"\n=== {lang} ===")
    display(Audio(out))
    files.download(out)

print("\nIf English now sounds like your good English accent and Hindi like your Hindi — the split fixed it.")